# CNN Advisor Metric Audit + Colab GPU Runner

This notebook runs the standalone CNN-advisor evaluation on Colab GPU, with the corrected smoke-validity and metric-audit payload.

Guardrail: keep this workflow isolated from production submission paths (run.py, kernel default runners).

In [ ]:
# Optional: mount Drive if you want persistent outputs
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set your repo source.
# Option A: public/private git URL (preferred when access is configured).
REPO_URL = 'https://github.com/drosadocastro-bit/Project-Atabey.git'
REPO_DIR = '/content/Project-Atabey'

# Option B: use an existing Drive checkout by setting REPO_DIR directly.
# REPO_DIR = '/content/drive/MyDrive/Project-Atabey'

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

%cd /content/Project-Atabey

In [ ]:
# Install project and CNN dependencies for Colab Linux runtime.
%pip install -q -r requirements.txt
%pip install -q -e .
%pip install -q cellpose stardist csbdeep tensorflow tensorboard

In [ ]:
# Sanity-check GPU visibility for both torch and tensorflow.
import torch
import tensorflow as tf

print('torch:', torch.__version__)
print('torch cuda available:', torch.cuda.is_available())
print('torch device count:', torch.cuda.device_count())
print('tensorflow:', tf.__version__)
print('tensorflow gpus:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Fast corrected smoke to verify metric payload shape and warnings.
!PYTHONPATH=src python scripts/run_cnn_advisor_evaluation.py \
  --train-dir train \
  --scan-json submissions/cfar_bounded_scan_fulltrain.json \
  --max-samples 3 \
  --max-timepoints 2 \
  --skip-cellpose \
  --skip-stardist \
  --output-json submissions/cnn_advisor_eval_metric_audit_smoke_colab.json \
  --output-summary-json submissions/cnn_advisor_eval_metric_audit_smoke_colab_summary.json

In [ ]:
# Evidentiary rerun (larger cohort) with GPU-enabled CNN methods.
!PYTHONPATH=src python scripts/run_cnn_advisor_evaluation.py \
  --train-dir train \
  --scan-json submissions/cfar_bounded_scan_fulltrain.json \
  --max-samples 12 \
  --max-timepoints 10 \
  --use-gpu \
  --cellpose-epochs 2 \
  --stardist-epochs 2 \
  --output-json submissions/cnn_advisor_eval_colab_gpu.json \
  --output-summary-json submissions/cnn_advisor_eval_colab_gpu_summary.json

In [ ]:
import json
from pathlib import Path

summary_path = Path('submissions/cnn_advisor_eval_colab_gpu_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2))
else:
    print('Summary file not found yet:', summary_path)

In [ ]:
# Optional: download outputs from Colab.
from pathlib import Path
from google.colab import files

for path in [
    'submissions/cnn_advisor_eval_metric_audit_smoke_colab.json',
    'submissions/cnn_advisor_eval_metric_audit_smoke_colab_summary.json',
    'submissions/cnn_advisor_eval_colab_gpu.json',
    'submissions/cnn_advisor_eval_colab_gpu_summary.json',
]:
    if Path(path).exists():
        files.download(path)